# 06 · Bonus: the grounded assistant, end to end

This notebook isn't graded. With goals 1–5 fixed, the whole system works together. `GraphGroundedAssistant.invoke` (`src/assistant.py`) runs these steps:

1. **Memory.** It reads the conversation history from DynamoDB (goal 1: oldest first).
2. **Governed metrics.** It resolves the question against the metric vector index in Neo4j (`resolve_governed_metric`).
3. **Documents.** It runs hybrid RRF retrieval in OpenSearch, grounded through the graph (goals 3 and 4).
4. **Hallucination gate.** It **refuses** if neither a metric nor a document was retrieved.
5. **Metric values.** It computes each resolved metric over the PostgreSQL source of record, using allow-listed SQL with bound parameters, read-only.
6. **Answer.** It asks Nova Pro (Bedrock Converse) to answer **strictly** from that context.

`invoke` reads `s-demo` but never writes to it, so it can't disturb the goal 1 evidence.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('.'))
import labkit
from src.assistant import GraphGroundedAssistant

agent = GraphGroundedAssistant()

def ask(question, session_id='s-demo'):
    print(f'Q: {question}\n')
    print('A:', agent.invoke(session_id, question), '\n' + '-' * 100)

ask('What are the insurance exclusions for Nexus Tower?')
ask('What is the Solvency II ratio for the commercial property line?')

The marine-cargo branch only exists because of goal 3. The metric resolves from the graph and its value comes from `cargo_experience` (0.62):

In [ ]:
ask('What is the loss ratio for marine cargo, and what does the Atlantic Freight policy exclude?')

### Look under the hood

This is the context the model was given for the last question: resolved metrics, scores and grounded entities.

In [ ]:
q = 'What is the loss ratio for marine cargo?'
labkit.pretty(agent.retriever.resolve_governed_metric(q, top_k=1, threshold=0.75))
labkit.pretty(agent.retriever.hybrid_rag_traversal(q, top_k=2))

### The refusal gate

A question unrelated to anything in the graph or the index may still retrieve *something*: kNN always returns its nearest neighbours. The metric threshold (0.75) and "answer strictly from context" are what keep the answer honest. Try a few:

In [ ]:
ask('What was the weather in Lisbon yesterday?', session_id='s-scratch-offtopic')

In [ ]:
agent.close()